# Práctica 1 · Paso 2 — Índices de extremos observados y registro de eventos

**Cambio Climático · Ingeniería en Riesgos de Desastres · ESPAM MFL** · Material CC-IRD-P1-02 · Versión 1.0 · octubre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/riesgos-desastres/practica-1/RD1_02_extremos_eventos.ipynb)

**Objetivo.** Calcular los índices de extremos de precipitación y temperatura de su zona, su tendencia, su relación con ENSO y sus periodos de retorno, y contrastarlos con el registro de desastres del territorio.

**Producto.** Tabla de índices anuales (ETCCDI), tendencias, periodos de retorno, lista de lluvias extremas candidatas y el registro de eventos con su lluvia asociada.

**Requisito.** Haber ejecutado, con la misma zona, los cuadernos comunes [`CR01_descarga_reanalisis`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR01_descarga_reanalisis.ipynb) y [`CR02_series_zona`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR02_series_zona.ipynb). Conexión a internet para traer los índices de ENSO de NOAA (queda una copia en su Drive). Haber ejecutado el paso 1 (usa sus índices de ENSO). Para el registro: `registro_eventos.csv` llenado por el equipo (sección 9).

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Elija sus datos en el formulario de la celda **1** (la misma provincia o cantón que en los cuadernos comunes `CR01` y `CR02`) y ejecute las demás en orden. Los dos integrantes del equipo ejecutan el cuaderno, cada uno en su Drive.

<details><summary>Secuencia de la práctica</summary>


| Orden | Cuaderno común (las dos carreras) | Qué hace | Deja listo en `Mi unidad/CC_datos/<zona>/` |
|---|---|---|---|
| C1 | [`CR01_descarga_reanalisis`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR01_descarga_reanalisis.ipynb) | Descarga ERA5 y CORe solo para su zona (opción A, con su cuenta) o los trae del release del curso (opción B) | Archivos NetCDF e inventario |
| C2 | [`CR02_series_zona`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR02_series_zona.ipynb) | Promedia sobre la zona, ponderando cada celda por su área dentro del límite | Series diaria y mensual |

| Orden | Riesgos de Desastres · Práctica 1 | Qué hace | Deja listo en `Mi unidad/CC_Riesgos/practica-1/<zona>/` |
|---|---|---|---|
| 1 | [`RD1_01_ciclo_anual_enso`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/riesgos-desastres/practica-1/RD1_01_ciclo_anual_enso.ipynb) | **Paso 1:** ciclo anual y compuestos por fase de ENSO | Tablas y figuras del paso 1 |
| 2 | [`RD1_02_extremos_eventos`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/riesgos-desastres/practica-1/RD1_02_extremos_eventos.ipynb) | **Paso 2:** índices de extremos observados y registro de eventos | Índices, periodos de retorno y registro contrastado |

</details>

© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

## 1. Datos del equipo y opciones

In [ ]:
# @title Datos del estudiante y zona de trabajo (la misma en todos los cuadernos)
ESTUDIANTE = "Nombre Apellido"  # @param {type:"string"}
NIVEL = "provincia"  # @param ["provincia", "cantón"]
PROVINCIA = "Manabí"  # @param ["Azuay", "Bolívar", "Cañar", "Carchi", "Chimborazo", "Cotopaxi", "El Oro", "Esmeraldas", "Galápagos", "Guayas", "Imbabura", "Loja", "Los Ríos", "Manabí", "Morona Santiago", "Napo", "Orellana", "Pastaza", "Pichincha", "Santa Elena", "Santo Domingo de los Tsáchilas", "Sucumbíos", "Tungurahua", "Zamora Chinchipe"]
CANTON = "Manabí · Portoviejo"  # @param ["Azuay · Camilo Ponce Enriquez", "Azuay · Chordeleg", "Azuay · Cuenca", "Azuay · El Pan", "Azuay · Giron", "Azuay · Guachapala", "Azuay · Gualaceo", "Azuay · Nabon", "Azuay · Oña", "Azuay · Paute", "Azuay · Pucara", "Azuay · San Fernando", "Azuay · Santa Isabel", "Azuay · Sevilla De Oro", "Azuay · Sigsig", "Bolívar · Caluma", "Bolívar · Chillanes", "Bolívar · Chimbo", "Bolívar · Echeandia", "Bolívar · Guaranda", "Bolívar · Las Naves", "Bolívar · San Miguel", "Cañar · Azogues", "Cañar · Biblian", "Cañar · Cañar", "Cañar · Deleg", "Cañar · El Tambo", "Cañar · La Troncal", "Cañar · Suscal", "Carchi · Bolivar", "Carchi · Espejo", "Carchi · Mira", "Carchi · Montufar", "Carchi · San Pedro De Huaca", "Carchi · Tulcan", "Chimborazo · Alausi", "Chimborazo · Chambo", "Chimborazo · Chunchi", "Chimborazo · Colta", "Chimborazo · Cumanda", "Chimborazo · Guamote", "Chimborazo · Guano", "Chimborazo · Pallatanga", "Chimborazo · Penipe", "Chimborazo · Riobamba", "Cotopaxi · La Mana", "Cotopaxi · Latacunga", "Cotopaxi · Pangua", "Cotopaxi · Pujili", "Cotopaxi · Salcedo", "Cotopaxi · Saquisili", "Cotopaxi · Sigchos", "El Oro · Arenillas", "El Oro · Atahualpa", "El Oro · Balsas", "El Oro · Chilla", "El Oro · El Guabo", "El Oro · Huaquillas", "El Oro · Las Lajas", "El Oro · Machala", "El Oro · Marcabeli", "El Oro · Pasaje", "El Oro · Piñas", "El Oro · Portovelo", "El Oro · Santa Rosa", "El Oro · Zaruma", "Esmeraldas · Atacames", "Esmeraldas · Eloy Alfaro", "Esmeraldas · Esmeraldas", "Esmeraldas · La Concordia", "Esmeraldas · Muisne", "Esmeraldas · Quininde", "Esmeraldas · Rioverde", "Esmeraldas · San Lorenzo", "Galápagos · Isabela", "Galápagos · San Cristobal", "Galápagos · Santa Cruz", "Guayas · Alfredo Baquerizo Moreno", "Guayas · Balao", "Guayas · Balzar", "Guayas · Colimes", "Guayas · Crnel. Marcelino Maridueña", "Guayas · Daule", "Guayas · Duran", "Guayas · El Triunfo", "Guayas · Empalme", "Guayas · Gnral. Antonio Elizalde", "Guayas · Guayaquil", "Guayas · Isidro Ayora", "Guayas · Lomas De Sargentillo", "Guayas · Milagro", "Guayas · Naranjal", "Guayas · Naranjito", "Guayas · Nobol", "Guayas · Palestina", "Guayas · Pedro Carbo", "Guayas · Playas", "Guayas · Salitre", "Guayas · Samborondon", "Guayas · San Jacinto De Yaguachi", "Guayas · Santa Lucia", "Guayas · Simon Bolivar", "Imbabura · Antonio Ante", "Imbabura · Cotacachi", "Imbabura · Ibarra", "Imbabura · Otavalo", "Imbabura · Pimampiro", "Imbabura · San Miguel De Urcuqui", "Loja · Calvas", "Loja · Catamayo", "Loja · Celica", "Loja · Chaguarpamba", "Loja · Espindola", "Loja · Gonzanama", "Loja · Loja", "Loja · Macara", "Loja · Olmedo", "Loja · Paltas", "Loja · Pindal", "Loja · Puyango", "Loja · Quilanga", "Loja · Saraguro", "Loja · Sozoranga", "Loja · Zapotillo", "Los Ríos · Baba", "Los Ríos · Babahoyo", "Los Ríos · Buena Fe", "Los Ríos · Mocache", "Los Ríos · Montalvo", "Los Ríos · Palenque", "Los Ríos · Puebloviejo", "Los Ríos · Quevedo", "Los Ríos · Quinsaloma", "Los Ríos · Urdaneta", "Los Ríos · Valencia", "Los Ríos · Ventanas", "Los Ríos · Vinces", "Manabí · 24 De Mayo", "Manabí · Bolivar", "Manabí · Chone", "Manabí · El Carmen", "Manabí · Flavio Alfaro", "Manabí · Jama", "Manabí · Jaramijo", "Manabí · Jipijapa", "Manabí · Junin", "Manabí · Manta", "Manabí · Montecristi", "Manabí · Olmedo", "Manabí · Pajan", "Manabí · Pedernales", "Manabí · Pichincha", "Manabí · Portoviejo", "Manabí · Puerto Lopez", "Manabí · Rocafuerte", "Manabí · San Vicente", "Manabí · Santa Ana", "Manabí · Sucre", "Manabí · Tosagua", "Morona Santiago · Gualaquiza", "Morona Santiago · Huamboya", "Morona Santiago · Limon Indanza", "Morona Santiago · Logroðo", "Morona Santiago · Morona", "Morona Santiago · Pablo Sexto", "Morona Santiago · Palora", "Morona Santiago · San Juan Bosco", "Morona Santiago · Santiago", "Morona Santiago · Sucua", "Morona Santiago · Taisha", "Morona Santiago · Tiwintza", "Napo · Archidona", "Napo · Carlos Julio Arosemena Tola", "Napo · El Chaco", "Napo · Quijos", "Napo · Tena", "Orellana · Aguarico", "Orellana · La Joya De Los Sachas", "Orellana · Loreto", "Orellana · Orellana", "Pastaza · Arajuno", "Pastaza · Mera", "Pastaza · Pastaza", "Pastaza · Santa Clara", "Pichincha · Cayambe", "Pichincha · Mejia", "Pichincha · Pedro Moncayo", "Pichincha · Pedro Vicente Maldonado", "Pichincha · Puerto Quito", "Pichincha · Quito", "Pichincha · Rumiðahui", "Pichincha · San Miguel De Los Bancos", "Santa Elena · La Libertad", "Santa Elena · Salinas", "Santa Elena · Santa Elena", "Santo Domingo de los Tsáchilas · Santo Domingo", "Sucumbíos · Cascales", "Sucumbíos · Cuyabeno", "Sucumbíos · Gonzalo Pizarro", "Sucumbíos · Lago Agrio", "Sucumbíos · Putumayo", "Sucumbíos · Shushufindi", "Sucumbíos · Sucumbios", "Tungurahua · Ambato", "Tungurahua · Baños De Agua Santa", "Tungurahua · Cevallos", "Tungurahua · Mocha", "Tungurahua · Patate", "Tungurahua · Quero", "Tungurahua · San Pedro De Pelileo", "Tungurahua · Santiago De Pillaro", "Tungurahua · Tisaleo", "Zamora Chinchipe · Centinela Del Condor", "Zamora Chinchipe · Chinchipe", "Zamora Chinchipe · El Pangui", "Zamora Chinchipe · Nangaritza", "Zamora Chinchipe · Palanda", "Zamora Chinchipe · Paquisha", "Zamora Chinchipe · Yacuambi", "Zamora Chinchipe · Yantzaza", "Zamora Chinchipe · Zamora"]
# NIVEL = "provincia": se usa PROVINCIA y NO se usa CANTON.
# NIVEL = "cantón": se usa CANTON (la lista está ordenada por provincia) y NO se usa PROVINCIA:
#                   la provincia sale del propio cantón.
if NIVEL == "cantón":
    _provincia_formulario = PROVINCIA
    PROVINCIA, CANTON = CANTON.split(" · ")
    print(f"Zona elegida: cantón {CANTON}, provincia de {PROVINCIA}.")
    if _provincia_formulario != PROVINCIA:
        print(f"  Aviso: el campo PROVINCIA dice {_provincia_formulario}, pero no se usa en el nivel cantón.")
else:
    print(f"Zona elegida: provincia de {PROVINCIA}. El campo CANTON no se usa en el nivel provincia.")
    CANTON = None

BASE_INI, BASE_FIN = 1981, 2010     # periodo base de los percentiles (ETCCDI)
ARCHIVO_ESTACION = ""  # @param {type:"string"}
# Opcional: CSV de una estación con el formato de practicas/comun/estaciones/plantilla_estacion.csv (vacío = sin estación)
ARCHIVO_EVENTOS = "registro_eventos.csv"  # @param {type:"string"}
# CSV del registro de eventos (sección 9). Si no existe, se crea una plantilla vacía en su carpeta de resultados.

## 2. Entorno de trabajo

In [ ]:
import os, sys, pathlib, subprocess, datetime, platform, json
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import limites, variables as V
from herramientas.reanalisis import era5, core, release
from herramientas.estaciones import calidad

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(".")

import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, geopandas as gpd
Z = limites.zona(PROVINCIA, CANTON)
GEOM, S = Z["geom"], Z["slug"]
DATOS = BASE / "CC_datos" / S            # datos y series comunes a las dos carreras (CR01, CR02)
RES = BASE / "CC_Riesgos" / "practica-1" / S                        # productos de este cuaderno
for c in (DATOS, RES): c.mkdir(parents=True, exist_ok=True)
print(f"Zona: {Z['nombre']} ({Z['tipo']}, DPA {Z['codigo']}, región {Z['region']})")
print(f"Datos en {DATOS} · resultados en {RES}")

from herramientas.indices import enso, extremos as X
import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning)

## 3. Las series diarias

Para los índices de extremos se necesitan datos **diarios**: ERA5 y CORe del cuaderno común (CORe diario solo si lo descargó con `CORE_DIARIO = True` o del release) y, si la tiene, una estación.

**Ojo con la escala.** Las series de la zona son promedios de área: un aguacero de 150 mm en una parroquia se diluye en el promedio de la provincia. Los índices de la zona sirven para tendencias y para comparar años; los valores absolutos de un sitio vienen de estaciones (o de la celda de ERA5 más cercana, sección 7).

In [ ]:
D = pd.read_csv(DATOS / f"serie_diaria_{S}.csv", index_col="fecha", parse_dates=True)
SERIES = {f: {v: D[f"{f}_{v}"] for v in ("pr", "tmax", "tmin") if f"{f}_{v}" in D} for f in ("ERA5", "CORe") if f"{f}_pr" in D}
if ARCHIVO_ESTACION:
    est, info = calidad.leer(ARCHIVO_ESTACION)
    est, _ = calidad.limites_fisicos(est, True)
    SERIES["Estación"] = {k: est[c] for k, c in (("pr", "precip"), ("tmax", "tmax"), ("tmin", "tmin")) if est[c].notna().sum() > 365}
FUENTES = list(SERIES)
COL = {"ERA5": "#C2531F", "CORe": "#4C78A8", "Estación": "#12263A"}
for f, s in SERIES.items():
    p = s["pr"]
    print(f"{f:9s} {p.first_valid_index():%Y-%m-%d} a {p.last_valid_index():%Y-%m-%d} · variables {list(s)} · días sin dato de lluvia: {int(p.isna().sum())}")
INFO_ENSO = pd.read_csv(RES / "RD1_01_indices_enso_mensual.csv", index_col=0, parse_dates=True)
meta1 = json.loads((RES / "RD1_01_metadatos.json").read_text(encoding="utf-8"))
INDICE_ENSO, MESES_T = meta1["indice_enso"], {"dic–may": [12, 1, 2, 3, 4, 5], "ene–abr": [1, 2, 3, 4],
    "oct–may": [10, 11, 12, 1, 2, 3, 4, 5], "anual": list(range(1, 13))}[meta1["temporada"]]
print("ENSO del paso 1:", INDICE_ENSO, "· temporada", meta1["temporada"])

## 4. Índices anuales (ETCCDI)

Definiciones del grupo de expertos ETCCDI (OMM/CLIVAR). Día lluvioso: P ≥ 1 mm. Percentiles en la base 1981–2010. Un año queda vacío si le faltan más de 15 días.

In [ ]:
IND = {}
for f, s in SERIES.items():
    partes = [X.precipitacion(s["pr"], (BASE_INI, BASE_FIN))]
    if {"tmax", "tmin"} <= set(s): partes.append(X.temperatura(s["tmax"], s["tmin"], (BASE_INI, BASE_FIN)))
    IND[f] = pd.concat(partes, axis=1)
    print(f"{f}: P95 = {partes[0].attrs['umbrales']['P95_mm']} mm/día, P99 = {partes[0].attrs['umbrales']['P99_mm']} mm/día")
TABLA = pd.concat({f: i.mean() for f, i in IND.items()}, axis=1).round(1)
TABLA.insert(0, "unidad", [X.DESCRIPCION[k][1] for k in TABLA.index]); TABLA.insert(0, "índice", [X.DESCRIPCION[k][0] for k in TABLA.index])
display(TABLA)

## 5. Tendencias

**Pendiente de Sen** (cambio por década, robusta frente a valores atípicos) y **prueba de Mann-Kendall** (¿la tendencia es distinguible del azar?). Con 40 años y un ENSO fuerte, una tendencia no significativa no prueba que no haya cambio: la variabilidad puede ocultarlo. Los años de El Niño fuerte pesan mucho en las tendencias de lluvia de la Costa; compruébelo quitándolos.

In [ ]:
TEND = pd.DataFrame({(f, k): X.tendencia(i[k]) for f, i in IND.items() for k in i.columns}).T
display(TEND.unstack(0).xs("sen_por_decada", axis=1, level=0).join(
        TEND.unstack(0).xs("p_valor", axis=1, level=0), lsuffix=" · Sen/década", rsuffix=" · p"))
PANEL = [k for k in ("Rx1day", "Rx5day", "R95pTOT", "CDD", "TX90p", "TN90p") if k in IND[FUENTES[0]]]
fig, ax = plt.subplots(2, 3, figsize=(15, 7)); ax = ax.ravel()
for a, k in zip(ax, PANEL):
    for f, i in IND.items():
        if k not in i: continue
        s = i[k].dropna(); a.plot(s.index, s, "o-", ms=3, lw=.8, color=COL[f], label=f)
        t = X.tendencia(s)
        if t["n"] >= 10:
            from scipy.stats import theilslopes
            m, b0 = theilslopes(s.values, s.index.values)[:2]
            a.plot(s.index, b0 + m * s.index.values, "--", color=COL[f], lw=1.2)
    a.set_title(f"{k} ({X.DESCRIPCION[k][1]})", fontsize=10); a.legend(fontsize=7)
for a in ax[len(PANEL):]: a.axis("off")
fig.suptitle(f"{Z['nombre']}: índices de extremos y tendencia de Sen"); fig.tight_layout()
fig.savefig(RES / f"RD1_02_tendencias_{S}.png", dpi=300); plt.show()

## 6. Extremos y ENSO

Cada año se clasifica por el índice de ENSO medio de su temporada lluviosa (el del paso 1). Si los extremos de lluvia de su zona dependen de El Niño, las cajas de "El Niño" estarán claramente más arriba.

In [ ]:
idx_t = enso.temporada(INFO_ENSO[INDICE_ENSO].dropna(), MESES_T, "mean")
u_c = enso.UMBRALES_ICEN["cálida débil"] if INDICE_ENSO == "ICEN" else enso.UMBRAL_ONI
u_f = enso.UMBRALES_ICEN["fría débil"] if INDICE_ENSO == "ICEN" else -enso.UMBRAL_ONI
FASE_ANIO = idx_t.map(lambda v: "El Niño" if v >= u_c else ("La Niña" if v < u_f else "Neutro"))
f0 = FUENTES[0]
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a, k in zip(ax, ("Rx1day", "Rx5day", "R95p")):
    d = IND[f0][[k]].join(FASE_ANIO.rename("fase"), how="inner").dropna()
    grupos = [d.loc[d.fase == fa, k] for fa in ("La Niña", "Neutro", "El Niño")]
    a.boxplot(grupos); a.set_xticks([1, 2, 3], [f"{fa}{chr(10)}n={len(g_)}" for fa, g_ in zip(("La Niña", "Neutro", "El Niño"), grupos)])
    a.set_title(f"{f0}: {k} (mm)")
fig.suptitle(f"Extremos de lluvia por fase de ENSO ({INDICE_ENSO}, temporada)"); fig.tight_layout()
fig.savefig(RES / f"RD1_02_extremos_enso_{S}.png", dpi=300); plt.show()

## 7. Periodos de retorno

Se ajusta una distribución de valores extremos generalizada (GEV) a los máximos anuales de 1 y 5 días. El **nivel de retorno de T años** es el valor que se supera en promedio una vez cada T años, es decir, con probabilidad 1/T en un año cualquiera. El intervalo sale de remuestrear los años: con ~40 años de datos, los periodos mayores de 50 años son extrapolación y se reportan con su incertidumbre.

Se agrega la **celda de ERA5 más cercana al centro de la zona**, que conserva mejor la intensidad local que el promedio de área.

In [ ]:
inv = json.loads((DATOS / "inventario.json").read_text(encoding="utf-8"))
ED = xr.open_dataset(DATOS / inv["archivos"]["era5_diario"])["pr"]
c = GEOM.representative_point()
la, lo = xr.broadcast(ED["lat"], ED["lon"])
dist = np.hypot(la - c.y, lo - c.x).where(ED.notnull().any("time"))
i_, j_ = np.unravel_index(int(np.nanargmin(dist.values)), dist.shape)
CELDA = ED.isel(lat=i_, lon=j_).to_series()
MAX = {f"{f} zona": IND[f]["Rx1day"] for f in IND}
MAX["ERA5 celda"] = X.precipitacion(CELDA, (BASE_INI, BASE_FIN))["Rx1day"]
MAX5 = {f"{f} zona": IND[f]["Rx5day"] for f in IND}
RET = {}
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
for a, (nombre, dic) in zip(ax, (("Rx1day", MAX), ("Rx5day", MAX5))):
    for k, s in dic.items():
        if s.notna().sum() < X.MIN_ANIOS_RETORNO:
            print(f"{nombre} · {k}: {s.notna().sum()} años, insuficientes para periodos de retorno (mínimo {X.MIN_ANIOS_RETORNO})"); continue
        r = X.retorno(s); RET[(nombre, k)] = r
        col = COL.get(k.split()[0], "k")
        a.plot(r.index, r["nivel"], "o-" if "zona" in k else "s--", color=col, label=k)
        a.fill_between(r.index, r["inf_5%"], r["sup_95%"], color=col, alpha=.12)
        emp = s.dropna().sort_values(ascending=False); T = (len(emp) + 1) / np.arange(1, len(emp) + 1)
        a.plot(T, emp.values, ".", color=col, alpha=.6)
    a.set_xscale("log"); a.set_xlabel("Periodo de retorno (años)"); a.set_ylabel("mm"); a.set_title(nombre); a.legend(fontsize=8)
fig.suptitle(f"{Z['nombre']}: niveles de retorno (GEV) · celda ERA5 {float(ED.lat[i_]):+.2f}, {float(ED.lon[j_]):+.2f}"); fig.tight_layout()
fig.savefig(RES / f"RD1_02_retorno_{S}.png", dpi=300); plt.show()
RETORNO = pd.concat(RET, names=["indice", "serie"]).round(1)
display(RETORNO.unstack([0, 1]).xs("nivel", axis=1, level=0))

## 8. Lluvias extremas candidatas

Las fechas con los mayores acumulados de 1 y 5 días, separadas al menos 10 días (eventos independientes), con la fase de ENSO de ese mes. **Son las fechas que conviene buscar primero** en los registros de desastres: si una lluvia de periodo de retorno alto no tiene registro, puede ser un vacío del registro o una zona poco expuesta; si un desastre no coincide con lluvia extrema, puede depender de la exposición o de la escala local.

In [ ]:
CAND = []
for f, s in SERIES.items():
    for d in (1, 5):
        t = X.mayores(s["pr"], 15, d)
        t["serie"], t["dias"] = f, d
        t["periodo_retorno_anios"] = [round(X.periodo_empirico(v, IND[f]["Rx1day" if d == 1 else "Rx5day"]), 1) for v in t[f"P{d}d_mm"]]
        t["fase_ENSO"] = [INFO_ENSO[f"fase_{INDICE_ENSO}"].get(pd.Timestamp(x).replace(day=1), "sin dato") for x in t["fecha_fin"]]
        CAND.append(t.rename(columns={f"P{d}d_mm": "P_mm"}).round({"P_mm": 1}))
CAND = pd.concat(CAND, ignore_index=True)
display(CAND[(CAND.serie == FUENTES[0]) & (CAND.dias == 5)])
print("% de las 15 mayores lluvias de 5 días en fase El Niño:",
      CAND[CAND.dias == 5].groupby("serie")["fase_ENSO"].apply(lambda x: round(100 * (x == "El Niño").mean())).to_dict())

## 9. Registro de eventos

El equipo registra los desastres de origen hidrometeorológico de su zona en `registro_eventos.csv`, **una fila por evento y con su fuente**. Fuentes sugeridas: la base DesInventar del Ecuador, los informes de situación de la Secretaría Nacional de Gestión de Riesgos, registros de los GAD y prensa local con fecha. Empiece por las fechas candidatas de la sección 8 y por los eventos conocidos (1982–83, 1997–98, 2008, 2012, 2017, 2023 y el evento en curso).

| Columna | Contenido |
|---|---|
| `fecha_inicio`, `fecha_fin` | AAAA-MM-DD (si solo hay una fecha, repítala) |
| `tipo` | inundación, deslizamiento, aluvión, sequía, incendio forestal, vendaval, oleaje u otro |
| `canton`, `parroquia` | lugar afectado |
| `descripcion` | qué pasó, en una línea |
| `fallecidos`, `afectados`, `viviendas_afectadas` | números; vacío si no se reporta (nunca 0 por defecto) |
| `fuente`, `url` | de dónde sale el dato; obligatorio |

Para cada evento, el cuaderno calcula la lluvia de los 1, 3, 5, 10 y 30 días previos en cada fuente y **qué tan rara fue** (percentil entre todas las ventanas del registro); para sequías e incendios, los días secos y la anomalía de lluvia de los 90 días previos; y la fase de ENSO del mes.

In [ ]:
CAMPOS = ["fecha_inicio", "fecha_fin", "tipo", "canton", "parroquia", "descripcion", "fallecidos", "afectados",
          "viviendas_afectadas", "fuente", "url"]
ruta_ev = pathlib.Path(ARCHIVO_EVENTOS)
if not ruta_ev.exists(): ruta_ev = RES / ARCHIVO_EVENTOS
if not ruta_ev.exists():
    pd.DataFrame(columns=CAMPOS).to_csv(ruta_ev, index=False, encoding="utf-8")
    print(f"Se creó la plantilla vacía {ruta_ev}. Llénela (en Drive, con Hojas de cálculo, guardando como CSV) y vuelva a ejecutar.")
EV = pd.read_csv(ruta_ev, parse_dates=["fecha_inicio", "fecha_fin"])
faltan = [c for c in CAMPOS if c not in EV]
assert not faltan, f"Al registro le faltan columnas: {faltan}"
SECOS = ("sequía", "sequia", "incendio forestal", "incendio")
filas = []
for _, e in EV.iterrows():
    fila = {"fecha_inicio": e.fecha_inicio.date(), "tipo": e.tipo, "canton": e.canton, "fuente": e.fuente}
    fila[f"fase_{INDICE_ENSO}"] = INFO_ENSO[f"fase_{INDICE_ENSO}"].get(e.fecha_inicio.replace(day=1), "sin dato")
    fila[INDICE_ENSO] = INFO_ENSO[INDICE_ENSO].get(e.fecha_inicio.replace(day=1), np.nan)
    for f, s in SERIES.items():
        if e.fecha_inicio < s["pr"].first_valid_index() or e.fecha_inicio > s["pr"].last_valid_index(): continue
        r = X.secos_previos(s["pr"], e.fecha_inicio) if str(e.tipo).lower() in SECOS else X.acumulados(s["pr"], e.fecha_inicio)
        fila.update({f"{f} · {k}": v for k, v in r.items()})
    filas.append(fila)
CONTRASTE = pd.DataFrame(filas)
if len(CONTRASTE):
    display(CONTRASTE)
    for f in SERIES:
        k = f"{f} · P5d_pct"
        if k in CONTRASTE:
            v = CONTRASTE[k].dropna()
            print(f"{f}: {len(v)} eventos de lluvia; {int((v >= 95).sum())} ({100 * (v >= 95).mean():.0f} %) con lluvia de 5 días sobre el percentil 95")
else:
    print("Registro vacío: complete registro_eventos.csv.")

In [ ]:
# Línea de tiempo: lluvia diaria de la zona y eventos registrados
if len(EV):
    f0 = FUENTES[0]; p = SERIES[f0]["pr"]
    fig, ax = plt.subplots(2, 1, figsize=(14, 6.5), gridspec_kw={"height_ratios": [2, 1]})
    ax[0].plot(p.index, p, color=COL[f0], lw=.4)
    ax[0].axhline(X.umbral_lluvia(p, 95, (BASE_INI, BASE_FIN)), color="k", ls="--", lw=.7, label="P95 de días lluviosos")
    tipos = EV["tipo"].astype(str).str.lower().unique()
    marcas = dict(zip(tipos, "v^so*Dxph+"))
    for t_ in tipos:
        sub = EV[EV["tipo"].astype(str).str.lower() == t_]
        ax[0].plot(sub.fecha_inicio, [p.max() * 1.08] * len(sub), marcas[t_], ms=7, label=t_)
    ax[0].set_ylim(0, p.max() * 1.15)
    ax[0].set_ylabel("mm/día"); ax[0].legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1)); ax[0].set_title(f"{f0}: lluvia diaria de la zona y eventos registrados")
    por_anio = EV.groupby(EV.fecha_inicio.dt.year).size()
    ax[1].bar(por_anio.index, por_anio.values, color="0.4", label="eventos registrados")
    ax2 = ax[1].twinx(); ax2.plot(IND[f0].index, IND[f0]["R20mm"], "o-", color=COL[f0], ms=3, label="R20mm")
    ax[1].set_ylabel("eventos"); ax2.set_ylabel("días con P ≥ 20 mm"); ax[1].legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)
    fig.tight_layout(); fig.savefig(RES / f"RD1_02_eventos_{S}.png", dpi=300); plt.show()

## 10. Guardar

In [ ]:
pd.concat(IND, axis=1).round(2).to_csv(RES / f"RD1_02_indices_anuales_{S}.csv", encoding="utf-8")
TEND.to_csv(RES / f"RD1_02_tendencias_{S}.csv", encoding="utf-8")
RETORNO.to_csv(RES / f"RD1_02_periodos_retorno_{S}.csv", encoding="utf-8")
CAND.to_csv(RES / f"RD1_02_lluvias_candidatas_{S}.csv", index=False, encoding="utf-8")
if len(CONTRASTE): CONTRASTE.to_csv(RES / f"RD1_02_registro_contrastado_{S}.csv", index=False, encoding="utf-8")
meta = {"material": "CC-IRD-P1-02 v1.0", "equipo": ESTUDIANTE, "zona": Z["nombre"], "base_percentiles": [BASE_INI, BASE_FIN],
        "series": FUENTES, "estacion": ARCHIVO_ESTACION or None, "indice_enso": INDICE_ENSO,
        "simplificaciones": ["rachas dentro del año calendario", "percentiles de temperatura sin remuestreo en la base",
                             f"año faltante con más de {X.MAX_FALTANTES_ANIO} días sin dato", "series de zona = promedio de área"],
        "celda_era5": [float(ED.lat[i_]), float(ED.lon[j_])], "eventos_registrados": int(len(EV)),
        "fecha": datetime.date.today().isoformat()}
(RES / "RD1_02_metadatos.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(sorted(p.name for p in RES.glob("RD1_02_*")))

## 11. Preguntas para el capítulo 1

1. ¿Qué índices de extremos muestran tendencia significativa en su zona y en qué sentido? ¿Coinciden ERA5, CORe y la estación? ¿Cambia la tendencia si quita 1983 y 1998?
2. ¿Cuánto llueve en un día y en cinco días con un periodo de retorno de 10 y de 25 años? Compare la zona con la celda: ¿por qué difieren y cuál usaría para diseñar una alcantarilla?
3. ¿Qué parte de las mayores lluvias ocurrió en fase de El Niño? Relaciónelo con los compuestos del paso 1.
4. Del registro de eventos: ¿qué porcentaje de las inundaciones y deslizamientos coincidió con lluvias sobre el percentil 95? Explique al menos un evento con lluvia común (exposición, escala local, lluvia en la parte alta de la cuenca) y al menos una lluvia extrema sin registro.
5. ¿Qué amenaza hidrometeorológica es la principal en su zona y en qué meses y fases de ENSO se concentra? Esta respuesta abre el capítulo 2 (riesgo).

**Para el informe:** periodo base, simplificaciones de los índices, escala de las series (zona, celda, estación), fuentes del registro y sus vacíos.

**Citas.** Zhang, X. et al. (2011). Indices for monitoring changes in extremes based on daily temperature and precipitation data. *WIREs Climate Change*, 2, 851–870 · Klein Tank, A., Zwiers, F. y Zhang, X. (2009). *Guidelines on analysis of extremes in a changing climate in support of informed decisions for adaptation*. WCDMP-72, OMM · Coles, S. (2001). *An Introduction to Statistical Modeling of Extreme Values*. Springer · Sen, P. K. (1968). *J. Am. Stat. Assoc.*, 63, 1379–1389.